# PostgreSQL Environment Setup — Connection Test (Pipeline TEST Run)

**Request:** `notebook-requests/PostgreSQL Environment Setup.txt` (DBAS5115 exercise)

**What this notebook is:** an automated **test run** of the my-hub notebook-requests
pipeline, executed on 2026-10-30 at James's request to prove the pipeline can turn a
plain-text request into a generated, *executed* notebook. It follows the exercise's
workflow — set up a PostgreSQL environment, test the database connection with a SQL
script, and show the resulting table records.

**One substitution, stated plainly:** the exercise refers to a *provided* file,
`PostGres_testing.sql`. That file is **not in this repository** and was not in the
local course materials, so this test run uses a clearly-labelled substitute,
`request_results/PostGres_testing_SUBSTITUTE.sql`, written in the same shape
(DROP TABLE at the top, CREATE TABLE, INSERTs, final SELECT).

## Environment Setup Steps (from the exercise)

The exercise as written for the course:

1. Open this repository in the browser.
2. Launch the repository in a **GitHub Codespace** and open it with VS Code, as shown in class.
3. Follow the environment setup steps from class so the VS Code environment inside the
   Codespace is ready for PostgreSQL development.
4. Use the provided `PostGres_testing.sql` to test the database connection — **do NOT run
   the `DROP TABLE` command the first time** the script is run.
5. Screenshot the resulting table records into an image file in the repository called
   `successful_setup.bmp`, then commit and push the whole repository for credit.

**For this pipeline test run**, the Codespace database is replaced by a real, throwaway
**PostgreSQL 16.2** server started locally just for this test, and the "screenshot" step is
replaced by the executed cell outputs below — no image is fabricated. (For actual course
credit, the Codespace run and the screenshot still have to be James's own — see the note at
the end.)

## 1. Connect to PostgreSQL

The connection string comes from the `PG_TEST_URI` environment variable, the same pattern a
Codespace would use to hand a database address to code. `SELECT version()` proves we are
talking to a real PostgreSQL server, not a mock.

In [1]:
import os

import pandas as pd
import psycopg2

PG_URI = os.environ.get("PG_TEST_URI", "postgresql://postgres:@/postgres?host=/tmp/pgdata")

conn = psycopg2.connect(PG_URI)
cur = conn.cursor()
cur.execute("SELECT version();")
print("Connected to:")
print(cur.fetchone()[0])

Connected to:
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## 2. Load the Test Script and Split It into Statements

The substitute script is read from `request_results/` and split into individual statements
so the first-run rule (skip the `DROP TABLE`) can be followed exactly, statement by statement.

In [2]:
SQL_PATH = "../request_results/PostGres_testing_SUBSTITUTE.sql"

sql_text = open(SQL_PATH, encoding="utf-8").read()

# Drop the comment lines, then split the remaining SQL on ';'
body = "\n".join(line for line in sql_text.splitlines() if not line.strip().startswith("--"))
statements = [s.strip() for s in body.split(";") if s.strip()]

for i, stmt in enumerate(statements, 1):
    first_line = stmt.splitlines()[0]
    print(f"{i}. {first_line}")

1. DROP TABLE IF EXISTS setup_test_records
2. CREATE TABLE setup_test_records (
3. INSERT INTO setup_test_records (full_name, course_code, task) VALUES
4. SELECT record_id, full_name, course_code, task, status, recorded_on


## 3. First Run — DROP TABLE Skipped, as Instructed

The exercise says not to run the `DROP TABLE` command the first time. On a first run the
table does not exist yet anyway; here the DROP statement is detected and skipped
explicitly, and every other statement runs in order.

In [3]:
cur = conn.cursor()
for stmt in statements:
    if stmt.upper().startswith("DROP"):
        print("SKIPPED on first run (per the exercise instructions):", stmt.splitlines()[0])
        continue
    cur.execute(stmt)
    print("Ran:", stmt.splitlines()[0])
conn.commit()
print("First run committed.")

SKIPPED on first run (per the exercise instructions): DROP TABLE IF EXISTS setup_test_records
Ran: CREATE TABLE setup_test_records (
Ran: INSERT INTO setup_test_records (full_name, course_code, task) VALUES
Ran: SELECT record_id, full_name, course_code, task, status, recorded_on
First run committed.


## 4. The Resulting Table Records

This is the output the exercise asks to screenshot: the records now stored in the test
table, read back from the database.

In [4]:
cur.execute(
    "SELECT record_id, full_name, course_code, task, status, recorded_on "
    "FROM setup_test_records ORDER BY record_id;"
)
rows = cur.fetchall()
columns = [desc[0] for desc in cur.description]
df_first = pd.DataFrame(rows, columns=columns)
df_first

,record_id,full_name,course_code,task,status,recorded_on
0,1,James Song,DBAS5115,PostgreSQL environment setup,complete,2026-10-30
1,2,James Song,DBAS5115,Database connection test,complete,2026-10-30
2,3,Ada Lovelace,DBAS5115,CREATE TABLE practice,complete,2026-10-30
3,4,Alan Turing,DBAS5115,INSERT practice,complete,2026-10-30
4,5,Grace Hopper,DBAS5115,SELECT practice,complete,2026-10-30


## 5. Second Run — Now With the DROP TABLE

With the first run done, the full script — including the `DROP TABLE` at the top — can be
run safely. This proves the script is re-runnable: the table is dropped, recreated, and
loaded again, and the same records come back.

In [5]:
cur = conn.cursor()
for stmt in statements:
    cur.execute(stmt)
    print("Ran:", stmt.splitlines()[0])
conn.commit()

cur.execute(
    "SELECT record_id, full_name, course_code, task, status, recorded_on "
    "FROM setup_test_records ORDER BY record_id;"
)
rows = cur.fetchall()
df_second = pd.DataFrame(rows, columns=[desc[0] for desc in cur.description])
print("Second run committed.")
df_second

Ran: DROP TABLE IF EXISTS setup_test_records
Ran: CREATE TABLE setup_test_records (
Ran: INSERT INTO setup_test_records (full_name, course_code, task) VALUES
Ran: SELECT record_id, full_name, course_code, task, status, recorded_on
Second run committed.


,record_id,full_name,course_code,task,status,recorded_on
0,1,James Song,DBAS5115,PostgreSQL environment setup,complete,2026-10-30
1,2,James Song,DBAS5115,Database connection test,complete,2026-10-30
2,3,Ada Lovelace,DBAS5115,CREATE TABLE practice,complete,2026-10-30
3,4,Alan Turing,DBAS5115,INSERT practice,complete,2026-10-30
4,5,Grace Hopper,DBAS5115,SELECT practice,complete,2026-10-30


## 6. What This Proves — and What It Does Not

**Proven by this test run:**

- A real PostgreSQL server accepted a connection from this notebook.
- The (substitute) test script created a table, inserted 5 records, and a SELECT read
  them back — shown as executed output above, first with the DROP skipped, then in full.
- The my-hub pipeline handled a real request end to end: text file in `notebook-requests/`
  → generated, executed notebook in `notebooks/`, supporting files in `request_results/`.

**Not a course submission:** the provided `PostGres_testing.sql` was missing, so a
substitute script was used, and the database here was a local test instance — not James's
Codespace. For course credit, James still needs to run the original file in his own
Codespace and save **his own** `successful_setup.bmp` screenshot. No screenshot was
fabricated in this test run.

In [6]:
conn.close()
print("Connection closed.")

Connection closed.